# 공간/강건성 VLM 실습

## 목표

- label map에서 경계를 계산합니다.
- 이벤트 카메라의 brightness-change stream을 toy data로 만듭니다.
- 이미지 증거와 텍스트 오정보가 충돌할 때 판단 규칙을 만들어 봅니다.


In [ ]:
labels = [
    [0, 0, 1, 1],
    [0, 2, 2, 1],
    [2, 2, 2, 1],
]

def boundary_map(grid):
    h, w = len(grid), len(grid[0])
    out = [[0 for _ in range(w)] for _ in range(h)]
    for r in range(h):
        for c in range(w):
            here = grid[r][c]
            right = c + 1 < w and grid[r][c + 1] != here
            down = r + 1 < h and grid[r + 1][c] != here
            out[r][c] = 1 if right or down else 0
    return out

for row in boundary_map(labels):
    print("".join("#" if x else "." for x in row))


In [ ]:
frame_a = [[10, 10, 10], [10, 30, 10], [10, 10, 10]]
frame_b = [[10, 12, 10], [10, 80, 10], [10, 11, 10]]

def event_stream(prev, curr, threshold=20):
    events = []
    for r in range(len(prev)):
        for c in range(len(prev[0])):
            diff = curr[r][c] - prev[r][c]
            if abs(diff) >= threshold:
                events.append({"row": r, "col": c, "polarity": 1 if diff > 0 else -1})
    return events

print(event_stream(frame_a, frame_b))


In [ ]:
def answer_with_modality_check(image_evidence, text_claim):
    """이미지 증거와 텍스트 주장이 충돌하면 답변을 보류합니다."""
    if image_evidence != text_claim:
        return "conflict: verify image evidence first"
    return "consistent: answer can use both modalities"

print(answer_with_modality_check("red car", "blue car"))
print(answer_with_modality_check("red car", "red car"))
